# World trade network Data processing

After downloading all datasets (as described in raw/worldtrade/readme.txt) we proceed to process the data. 

We need to build the adjacency matrices for each snapshot.

In [1]:
import pandas as pd
import warnings
import xlrd
import numpy as np
import seaborn as sns
warnings.filterwarnings('ignore')

In [2]:
input_folder = "data/raw/worldtrade/"
output_folder = "data/processed/worldtrade/"

!mkdir -p data/processed/worldtrade/

### Adjacency matrices

We want to build the adjacency matrices. $A^t_{ij}$ corresponds to the trade between country $i$ and $j$ at snapshot $t$. We store each snapshot independently in our output folder. We only consider even years from 2002 to 2023. 

Note that the values are stored as floats and the units are thousands of USD.

Because the dataset contains one entry per product, we sum the trade values for each importer-exporter pair over all products.

In [3]:
aggregated_trade = pd.DataFrame(columns=["year", "exporter", "importer", "value"])

#aggregated trade will contain all trades for each EVEN YEAR:
for year in [str(y) for y in range(2002, 2023, 2)]:
    trade_thisyear = pd.read_csv(f"{input_folder}/BACI_HS02_Y{year}_V202401b.csv")
    trade_thisyear.columns = ["year", "exporter", "importer", "product", "value", "quantity"]
    trade_thisyear = trade_thisyear.drop(["product", "quantity"], axis=1)
    trade_thisyear = trade_thisyear.groupby(["year", "exporter", "importer"]).sum().reset_index()
    aggregated_trade = pd.concat([aggregated_trade, trade_thisyear])

In [4]:
aggregated_trade.head()

,year,exporter,importer,value
0,2002,4,12,2295.157
1,2002,4,20,12.5
2,2002,4,36,102.307
3,2002,4,40,147.489
4,2002,4,48,73.562


In [6]:
#to convert country codes to country names (ISO-3)...
country_conversion = pd.read_csv(f"{input_folder}/country_codes_V202401b.csv")
country_conversion = dict(zip(country_conversion["country_code"], country_conversion["country_iso3"]))
aggregated_trade["exporter"] = [country_conversion[exp] for exp in aggregated_trade["exporter"]]
aggregated_trade["importer"] = [country_conversion[imp] for imp in aggregated_trade["importer"]]
aggregated_trade.head()

,year,exporter,importer,value
0,2002,AFG,DZA,2295.157
1,2002,AFG,AND,12.5
2,2002,AFG,AUS,102.307
3,2002,AFG,AUT,147.489
4,2002,AFG,BHR,73.562


In [7]:
#set of countries
countries = sorted(set(aggregated_trade["importer"]).union(set(aggregated_trade["exporter"])))
print(f"N={len(countries)}")

N=229


In [ ]:
#let's finally build the adjacency matrices with the same set of countries:

index_country = dict(zip(countries, [i for i in range(len(countries))])) #adjacency matrix index

for year in range(2002, 2023,2):
    this_adj = pd.DataFrame(0., columns=countries, index=countries)
    this_trade = aggregated_trade[aggregated_trade["year"]==year]
    #fill adjacency matrix with trade value
    for i in range(len(this_trade)):
        exp = this_trade.iloc[i, 1]
        imp = this_trade.iloc[i, 2]
        value = round(this_trade.iloc[i, 3], 3) #float precision: 3
        this_adj.iloc[index_country[exp], index_country[imp]] = value
    #save the matrices as csv
    this_adj.to_csv(f"{output_folder}/adjMat_{str(year)}.csv")